# PRACTICE 3: CLUSTERING RETAIL CUSTOMERS
## CHẶNG 1: KHÁM PHÁ DỮ LIỆU KHÁCH HÀNG BÁN LẺ (EXPLORATORY DATA ANALYSIS)

> **Mục tiêu kỹ thuật:**
> - Khảo sát bộ dữ liệu thô `Mall_Customers.csv` (200 khách hàng) theo nguyên tắc Read-Only 100%.
> - Khám phá cấu trúc bảng, kiểm định chất lượng và đo lường các chỉ số thống kê mô tả.
> - Trực quan hóa phân phối đơn biến (Tuổi, Thu nhập hàng năm, Điểm chi tiêu).
> - Phân tích hai biến: Tác động của Giới tính và Độ tuổi đến hành vi chi tiêu.
> - Khám phá bản đồ phân tán không gian 2D Thu nhập - Chi tiêu và ma trận tương quan Pearson.

---
## Mục 1: Khám Phá Cấu Trúc Bảng & Thống Kê Mô Tả Dữ Liệu Thô (Overview EDA)

### 1.1 Khai Báo Thư Viện & Cấu Hình Môi Trường Thực Nghiệm
* **Mục tiêu:** Cố định hạt giống ngẫu nhiên `RANDOM_SEED = 42`, cấu hình thư viện đồ họa Matplotlib/Seaborn và thiết lập đường dẫn gốc.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Đảm bảo import được các module từ thư mục gốc dự án
BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# Cấu hình thẩm mỹ biểu đồ
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']

### 1.2 Đọc Dữ Liệu Thô & Đánh Giá Cấu Trúc Bảng
* **Mục tiêu:** Nạp tệp `Mall_Customers.csv`, kiểm tra kích thước mẫu, kiểu dữ liệu từng cột và kiểm tra tính toàn vẹn.

In [ ]:
from scripts.common import RAW_CSV, to_rel_path

df_raw = pd.read_csv(RAW_CSV)
print(f"✔ Đã nạp dữ liệu từ: {to_rel_path(RAW_CSV)}")
print(f"• Quy mô bảng: {df_raw.shape[0]} khách hàng x {df_raw.shape[1]} thuộc tính.")
print(f"• Số lượng giá trị khuyết thiếu (NaN): {df_raw.isnull().sum().sum()} ô.")
display(df_raw.head())

### 1.3 Thống Kê Mô Tả & Cơ Cấu Giới Tính Khách Hàng
* **Mục tiêu:** Đo lường các chỉ số thống kê đặc trưng (Mean, Std, Min, Max, Tứ phân vị) và tỷ trọng giới tính.

In [ ]:
print("1. Bảng thống kê mô tả 3 biến số học:")
display(df_raw.describe().round(2))

gender_counts = df_raw['Gender'].value_counts()
gender_pct = (df_raw['Gender'].value_counts(normalize=True) * 100).round(2)
print("\n2. Cơ cấu giới tính khách hàng tại trung tâm thương mại:")
for g in gender_counts.index:
    print(f"  • {g}: {gender_counts[g]} khách ({gender_pct[g]}%)")

---
## Mục 2: Trực Quan Hóa Phân Phối Đơn Biến & Phân Tích Hai Biến

### 2.1 Khảo Sát Phân Phối Xác Suất Đơn Biến (KDE & Histogram)
* **Mục tiêu:** Khảo sát hình thái phân phối của 3 biến số: Độ tuổi, Thu nhập hàng năm và Điểm chi tiêu.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
num_features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
palette_colors = ['#3182ce', '#38a169', '#d69e2e']

for idx, (col, col_color) in enumerate(zip(num_features, palette_colors)):
    sns.histplot(df_raw[col], kde=True, ax=axes[idx], color=col_color, bins=15)
    axes[idx].set_title(f"Phân phối {col}", fontsize=11, fontweight='bold')
    axes[idx].set_xlabel(col)

plt.tight_layout()
plt.show()

### 2.2 So Sánh Mức Thu Nhập & Điểm Chi Tiêu Theo Giới Tính
* **Mục tiêu:** Sử dụng Boxplot để kiểm tra sự khác biệt về mức độ chi tiêu và thu nhập giữa Nam và Nữ.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='Gender', y='Annual Income (k$)', data=df_raw, ax=axes[0], palette=['#4299e1', '#ed64a6'])
axes[0].set_title("So sánh Thu nhập hàng năm theo Giới tính", fontsize=11, fontweight='bold')

sns.boxplot(x='Gender', y='Spending Score (1-100)', data=df_raw, ax=axes[1], palette=['#4299e1', '#ed64a6'])
axes[1].set_title("So sánh Điểm chi tiêu theo Giới tính", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

### 2.3 Phân Tích Xu Hướng Điểm Chi Tiêu Theo Nhóm Tuổi
* **Mục tiêu:** Đánh giá xu hướng biến thiên của điểm chi tiêu theo độ tuổi khách hàng bằng đường hồi quy tuyến tính.

In [ ]:
plt.figure(figsize=(10, 5))
sns.regplot(x='Age', y='Spending Score (1-100)', data=df_raw,
            color='#805ad5', scatter_kws={'alpha': 0.65, 's': 50}, line_kws={'color': '#e53e3e', 'linewidth': 2})
plt.title("Xu hướng Điểm chi tiêu theo Độ tuổi khách hàng", fontsize=12, fontweight='bold')
plt.xlabel("Độ tuổi (Age)")
plt.ylabel("Điểm chi tiêu (Spending Score 1-100)")
plt.show()

---
## Mục 3: Bản Đồ Phân Tán 2D Không Gian Gom Cụm & Ma Trận Tương Quan

### 3.1 Bản Đồ Phân Tán 2D Thu Nhập vs Điểm Chi Tiêu (Không Gian Vàng)
* **Mục tiêu:** Quan sát sự hội tụ tự nhiên của các điểm dữ liệu thành 5 phân cụm rõ rệt.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df_raw['Annual Income (k$)'], df_raw['Spending Score (1-100)'],
            color='#2b6cb0', edgecolors='white', s=70, alpha=0.85)
plt.title("Không gian Phân cụm 2D: Thu nhập hàng năm vs Điểm chi tiêu", fontsize=13, fontweight='bold')
plt.xlabel("Thu nhập hàng năm (Annual Income k$)", fontsize=11)
plt.ylabel("Điểm chi tiêu (Spending Score 1-100)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### 3.2 Ma Trận Hệ Số Tương Quan Tuyến Tính Pearson
* **Mục tiêu:** Đo lường mối liên hệ tuyến tính giữa các biến số, xác nhận tính độc lập trực giao giữa Thu nhập và Chi tiêu.

In [ ]:
plt.figure(figsize=(7, 5))
corr_matrix = df_raw[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='Blues', fmt='.4f', square=True, linewidths=0.5)
plt.title("Ma trận Hệ số Tương quan Pearson", fontsize=12, fontweight='bold')
plt.show()